# Phase 5: SAM-Guided Instance Segmentation & Defect Measurement
### Capstone Project: Physical Geometry Analysis for IPC Standards
**Hardware:** Windows 10/11 | NVIDIA RTX 4090 (24GB VRAM)  
**Tools:** SAM (Segment Anything Model), YOLO26-seg / YOLO11-seg, OpenCV  
**Goal:**
1. Upgrade from bounding boxes to pixel-accurate defect masks using SAM prompted by bounding boxes.
2. Train an instance segmentation model (YOLO-seg).
3. Compute physical geometry metrics:
   - Remaining trace width (IPC-A-610 standards compliance)
   - Open circuit gap distance ($\mu m$)
   - Spur protrusion surface area ($px^2$)


## 1. Environment & SAM Library Check


In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"CUDA Available: {torch.cuda.is_available()}")
# Check for segment-anything
try:
    from segment_anything import sam_model_registry, SamPredictor
    print("[+] Segment Anything (SAM) library loaded.")
except ImportError:
    print("[!] SAM library not installed. Install via: pip install git+https://github.com/facebookresearch/segment-anything.git")


## 2. Converting Bounding Boxes to Precise Masks via SAM
Prompt SAM with ground-truth bounding boxes to generate segmentation masks without manual pixel annotations.


In [ ]:
def generate_pseudo_mask(sam_predictor, image_rgb, bbox_xyxy):
    """Prompt SAM with a bounding box [x1, y1, x2, y2] to extract pixel mask."""
    sam_predictor.set_image(image_rgb)
    input_box = np.array(bbox_xyxy)
    masks, scores, _ = sam_predictor.predict(
        box=input_box,
        multimask_output=False
    )
    return masks[0], scores[0]

print("[+] Pseudo-mask generator defined.")


## 3. Train YOLO Instance Segmentation Model (YOLO-seg)
Train an instance segmentation model on the generated polygonal masks.


In [ ]:
from ultralytics import YOLO

# Segmentation model configuration
SEG_CONFIG = {
    "data": "pcb_seg.yaml", # Config file pointing to polygon annotations
    "epochs": 100,
    "imgsz": 640,
    "batch": 32,            # RTX 4090 memory optimization
    "device": 0,
    "amp": True,
    "patience": 20
}

# Initialize segmentation model
try:
    seg_model = YOLO("yolo26s-seg.pt")
except Exception:
    seg_model = YOLO("yolo11s-seg.pt")

print("[+] Segmentation model initialized. Ready for training.")
# To run training:
# seg_model.train(project="Phase5_Segmentation", name="YOLO_Seg_PCB", **SEG_CONFIG)


## 4. Physical Geometry Measurement Algorithm
Compute the exact trace width and gap distance to enforce factory quality thresholds.


In [ ]:
def measure_defect_geometry(defect_mask, pixel_to_micron_ratio=15.0):
    """Analyze a binary defect mask to extract geometric dimensions in microns."""
    contours, _ = cv2.findContours(defect_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return {"area_um2": 0, "max_dimension_um": 0}
        
    largest_cnt = max(contours, key=cv2.contourArea)
    area_px = cv2.contourArea(largest_cnt)
    
    # Minimum area bounding rectangle
    rect = cv2.minAreaRect(largest_cnt)
    (x, y), (w, h), angle = rect
    
    area_um2 = area_px * (pixel_to_micron_ratio ** 2)
    max_dim_um = max(w, h) * pixel_to_micron_ratio
    min_dim_um = min(w, h) * pixel_to_micron_ratio
    
    return {
        "area_um2": round(area_um2, 2),
        "length_um": round(max_dim_um, 2),
        "width_gap_um": round(min_dim_um, 2)
    }

# Test geometry measurement on simulated defect mask
test_mask = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(test_mask, (20, 40), (80, 48), 1, -1) # 60x8 px trace break
geom = measure_defect_geometry(test_mask, pixel_to_micron_ratio=10.0)
print(f"Calculated Defect Dimensions: {geom}")
